In [4]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
)

molecules_db = []

# --- Convert SMILES → 3D ---
def smiles_to_3d(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None

    mol = Chem.AddHs(mol)

    try:
        AllChem.EmbedMolecule(mol, AllChem.ETKDG())
        AllChem.UFFOptimizeMolecule(mol)
    except:
        return None, None

    mol_block = Chem.MolToMolBlock(mol)

    props = {
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "HDonors": Descriptors.NumHDonors(mol),
        "HAcceptors": Descriptors.NumHAcceptors(mol)
    }

    return mol_block, props


# --- Upload dataset ---
@app.post("/upload")
async def upload(file: UploadFile = File(...)):
    global molecules_db

    df = pd.read_csv(file.file)

    molecules_db = []

    for _, row in df.iterrows():
        smiles = row["canonical_smiles"]

        mol_block, props = smiles_to_3d(smiles)

        if mol_block:
            molecules_db.append({
                "smiles": smiles,
                "mol_block": mol_block,
                "props": props,
                "activity": row.get("standard_value", None)
            })

    return {"count": len(molecules_db)}


# --- Get molecules ---
@app.get("/molecules")
def get_molecules():
    return molecules_db


# --- Filter ---
@app.get("/filter")
def filter_molecules(min_logp: float = -10, max_logp: float = 10):
    return [
        m for m in molecules_db
        if min_logp <= m["props"]["LogP"] <= max_logp
    ]